# LX · Experiment 8 · Xu-inspired dual-adjacency source-space GCN (functional + atlas layers)

- **10 variant(s):** `plv__psd_de_strength`, `ciplv__psd_de_strength`, `plv__psd`, `ciplv__psd`, `plv__de`, `ciplv__de`, `plv__psd_de_strength__abl_functional_both`, `plv__psd_de_strength__abl_anatomical_both`, `plv__psd_de_strength__abl_bilateral_lobes`, `plv__psd_de_strength__abl_shared_weights`
- **Settings:** xu_dev_cv, xu_holdout (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 1 repeats × 5-fold subject-level outer CV; fixed hyperparameters, no hyperparameter search and no inner CV: one fit per outer training fold (protocol `gnn_fixed_hparams_v1`); 60 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** D3 LCMV beamforming + Xu et al. (2025) protocol (methodology log entry 50, `docs/lcmv_xu.md`). All outputs
are under `<outputs>/D3_LCMV_XU/`; D1/D2 and the earlier D3 (wMNE) runs and results are never touched.
**Validation (identical for Exp 0-7):** the frozen participant split saved with the dataset (`xu_split.json`):
`xu_dev_cv` = 5-fold participant-grouped CV inside the 80 % development participants; `xu_holdout` = fit on all
development participants, evaluated once on the 20 % holdout participants. Normalization, class weights and
(classical models only) the C selection use training participants only. Participant-level metrics (mean window
score per participant) are primary; window-level metrics are secondary.

**Runs both modes:** A (Xu tenfold, 650 samples; `configs/experiments/lx/tenfold/lx10_exp8_xu_dual_gcn.yaml`) first, then B (all accepted windows; `configs/experiments/lx/lx_exp8_xu_dual_gcn.yaml`).
Resumable: every completed fold is saved to Drive. Results: `<outputs>/D3_LCMV_XU/experiments_A_xu_tenfold/` and
`.../experiments_B_all_windows/`. Use a **GPU runtime** (T4).

In [ ]:
# ===== D3 LCMV / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_LX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The LCMV / Xu suite has its own code pin, <outputs>/D3_LCMV_XU/CODE_COMMIT.txt (all LX notebooks run
# that commit); the D1/D2 and D3 pins are only reported, never changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'LX']
     + (['--update-code'] if UPDATE_LX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
print('suite outputs:', LX)

## 0. This experiment's code pin
Exp 8 was added after Exp 0-7 had started on the suite pin (`D3_LCMV_XU/CODE_COMMIT.txt`). It runs the commit
recorded in `D3_LCMV_XU/lx_exp8_xu_dual_gcn_CODE_COMMIT.txt` (set on its first run; it never moves while Exp 8 has completed
work), so the suite pin and every completed Exp 0-7 result stay unchanged. Leave `UPDATE_EXP8_CODE = False`.

In [ ]:
UPDATE_EXP8_CODE = False
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
dirs = [Path(LX) / 'experiments_A_xu_tenfold' / 'lx10_exp8_xu_dual_gcn', Path(LX) / 'experiments_B_all_windows' / 'lx_exp8_xu_dual_gcn']
has_work = any(any(d.rglob('*.done.json')) for d in dirs if d.exists())
EXP_COMMIT, _ = resolve_code_pin(Path(LX) / 'lx_exp8_xu_dual_gcn_CODE_COMMIT.txt', head, update=UPDATE_EXP8_CODE, has_work=has_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', EXP_COMMIT])
for f in ('configs/experiments/lx/lx_exp8_xu_dual_gcn.yaml', 'src/adgraph/models/xu_dual.py', 'scripts/lx_exp8_inspect.py'):
    if not Path('/content/repo', f).exists():
        raise RuntimeError(f'pin {EXP_COMMIT[:10]} lacks {f}: set UPDATE_EXP8_CODE = True and run again')
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]                     # this notebook's own imports now come from Exp 8's commit
from adgraph.colab import sh
print('lx_exp8_xu_dual_gcn code commit:', EXP_COMMIT)
print('suite pin (Exp 0-7, unchanged):', (Path(LX) / 'CODE_COMMIT.txt').read_text().split()[0])

## 0b. Architecture check (no data): anatomical matrix, layers, parameter count per variant, one forward/backward pass on synthetic inputs of the real shapes

In [ ]:
sh("python scripts/lx_exp8_inspect.py")

## 1. Status (from the checkpoints on Drive)

In [ ]:
CONFIGS = {'A': 'configs/experiments/lx/tenfold/lx10_exp8_xu_dual_gcn.yaml', 'B': 'configs/experiments/lx/lx_exp8_xu_dual_gcn.yaml'}
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
for mode, cfg in CONFIGS.items():
    print('==== mode', mode)
    sh(f"python scripts/experiment.py status --config {cfg}")

## 2. Run mode A, then mode B (resumable)

In [ ]:
ERRORS = {}
for mode, cfg in CONFIGS.items():
    try:
        sh(f"python scripts/experiment.py run --config {cfg}")
    except RuntimeError as exc:
        ERRORS[mode] = exc
        print(f'mode {mode}: some units failed; details after the results')

## 3. Results (saved outputs only)

In [ ]:
import glob
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 300, 'display.width', 250)
for mode, cfg in CONFIGS.items():
    sh(f"python scripts/experiment.py report --config {cfg}")
    root = 'experiments_A_xu_tenfold' if mode == 'A' else 'experiments_B_all_windows'
    exp = Path(cfg).stem
    reports = sorted(glob.glob(f'{LX}/{root}/{exp}/_reports/report_*'))
    if reports:
        print('==== mode', mode, reports[-1])
        display(pd.read_csv(reports[-1] + '/results_summary.csv'))
        for png in sorted(glob.glob(reports[-1] + '/figures/*.png'))[:6]:
            display(Image(filename=png))
if ERRORS:
    raise RuntimeError(f'units did not complete in mode(s) {sorted(ERRORS)}; completed units are saved. Run all again.')

## 4. Comparison Exp 0-8 (all completed results of the suite; participant- and window-level metrics)

In [ ]:
sh("python scripts/lx_report.py")
rep = sorted(glob.glob(LX + '/reports/report_*'))[-1]
t = pd.read_csv(rep + '/comparison.csv')
cols = ['mode', 'experiment', 'variant', 'setting', 'n_participants', 'participant_accuracy',
        'participant_balanced_accuracy', 'participant_f1', 'participant_roc_auc', 'window_accuracy',
        'window_balanced_accuracy', 'window_f1', 'window_roc_auc', 'device']
display(t[[c for c in cols if c in t]].round(3)) if len(t) else print('no completed results yet')
for png in sorted(glob.glob(rep + '/figures/*.png')):
    display(Image(filename=png))